<a href="https://colab.research.google.com/github/springboardmentor12458j/LiveMeetingSummarize/blob/varshini/live_audio_transcription.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlit openai-whisper vosk pysoundfile librosa


In [ ]:
!apt-get update && apt-get install -y portaudio19-dev alsa-utils
!pip install pyaudio scipy


In [3]:
# Create transcription_service.py
transcription_service_code = '''
import whisper
import json
from vosk import Model, KaldiRecognizer
import soundfile as sf
import io

class TranscriptionService:
    def __init__(self):
        """Initialize transcription services"""
        try:
            self.whisper_model = whisper.load_model("base")
            self.vosk_model = Model(lang="en-us")
        except Exception as e:
            print(f"Warning: {e}")

    def transcribe_vosk(self, audio_path):
        """Transcribe using Vosk (Fast, Offline)"""
        try:
            data, samplerate = sf.read(audio_path)

            recognizer = KaldiRecognizer(self.vosk_model, samplerate)
            recognizer.SetWords(json.loads('[]'))

            # Process audio in chunks
            results = []
            chunk_size = 4000

            for i in range(0, len(data), chunk_size):
                chunk = (data[i:i+chunk_size] * 32768).astype('int16')
                recognizer.AcceptWaveform(chunk.tobytes())

            # Get final result
            final_result = json.loads(recognizer.FinalResult())
            text = final_result.get('result', '')

            if not text:
                text = final_result.get('partial', 'No speech detected')

            return ' '.join([word['conf'] for word in text if isinstance(text, list)]) or text
        except Exception as e:
            return f"Vosk Error: {str(e)}"

    def transcribe_whisper(self, audio_path):
        """Transcribe using OpenAI Whisper (High Quality)"""
        try:
            result = self.whisper_model.transcribe(audio_path, language="en")
            return result.get('text', 'No speech detected')
        except Exception as e:
            return f"Whisper Error: {str(e)}"
'''

with open('transcription_service.py', 'w') as f:
    f.write(transcription_service_code)

print("✅ TranscriptionService module created!")


✅ TranscriptionService module created!


In [4]:
import os
import json
from io import BytesIO
import soundfile as sf
import numpy as np
from IPython.display import Audio, display


In [5]:
from transcription_service import TranscriptionService

# Initialize the service
service = TranscriptionService()
print("✅ Transcription service initialized!")


100%|███████████████████████████████████████| 139M/139M [00:02<00:00, 66.5MiB/s]
vosk-model-small-en-us-0.15.zip: 100%|██████████| 39.3M/39.3M [00:03<00:00, 13.5MB/s]


✅ Transcription service initialized!


In [6]:
def record_audio_colab(duration=5, samplerate=16000):
    """
    Record audio in Google Colab (duration in seconds)
    """
    try:
        from google.colab import output
        from IPython.display import Javascript
        from base64 import b64decode
        import wave

        RECORD_SECONDS = duration

        js_code = Javascript('''
            async function recordAudio() {
                const stream = await navigator.mediaDevices.getUserMedia({audio: true});
                const mediaRecorder = new MediaRecorder(stream);
                let chunks = [];

                mediaRecorder.ondataavailable = e => chunks.push(e.data);
                mediaRecorder.start();

                await new Promise(r => setTimeout(r, %d * 1000));

                mediaRecorder.stop();
                const blob = new Blob(chunks, {'type': 'audio/wav'});
                const url = URL.createObjectURL(blob);
                const a = document.createElement('a');
                a.href = url;
                a.download = 'recording.wav';
                a.click();

                // Return audio data as base64
                const reader = new FileReader();
                reader.readAsArrayBuffer(blob);
                reader.onloadend = function() {
                    let binary = '';
                    const bytes = new Uint8Array(reader.result);
                    for (let i = 0; i < bytes.byteLength; i++) {
                        binary += String.fromCharCode(bytes[i]);
                    }
                    google.colab.kernel.invokeFunction('audio_data', [btoa(binary)], null);
                }
            }
            recordAudio();
        ''' % RECORD_SECONDS)

        display(js_code)
        print(f"⏳ Recording for {RECORD_SECONDS} seconds...")
    except Exception as e:
        print(f"Note: Audio recording requires browser permissions. Error: {e}")


In [7]:
# Option 1: Upload your own audio file
from google.colab import files

print("📁 Upload your audio file (WAV, MP3, etc.)")
uploaded_files = files.upload()

audio_path = list(uploaded_files.keys())[0] if uploaded_files else None

if audio_path:
    print(f"✅ File uploaded: {audio_path}")
else:
    print("⚠️ No file uploaded. Creating a test audio file...")
    # Create a sample audio for testing
    duration = 3
    samplerate = 16000
    t = np.linspace(0, duration, int(samplerate * duration))

    # Generate a simple test sound
    frequency = 440  # A4 note
    audio_data = 0.3 * np.sin(2 * np.pi * frequency * t)

    audio_path = "test_audio.wav"
    sf.write(audio_path, audio_data, samplerate)
    print(f"✅ Test audio created: {audio_path}")


📁 Upload your audio file (WAV, MP3, etc.)


Saving Live transcriber pro.mp3 to Live transcriber pro.mp3
✅ File uploaded: Live transcriber pro.mp3


In [8]:
# Display title
print("🎙️ Live Audio Transcriber")
print("=" * 50)
print("Record your audio and transcribe it using Vosk or Whisper.")
print("=" * 50)

if audio_path:
    # Model Selection
    print("\n🔄 Select Transcription Model:")
    print("1. Vosk (Fast, Offline)")
    print("2. Whisper (High Quality)")

    model_choice = input("Enter choice (1-2, default: 2): ").strip()

    # Transcribe
    print("\n⏳ Transcribing...")

    try:
        if model_choice == "1":
            transcribed_text = service.transcribe_vosk(audio_path)
            print("✅ Using Vosk Model")
        else:
            transcribed_text = service.transcribe_whisper(audio_path)
            print("✅ Using Whisper Model")

        # Display Result
        print("\n📝 Transcribed Text:")
        print("-" * 50)
        print(transcribed_text)
        print("-" * 50)

        # Save result to file
        with open("transcription_result.txt", "w") as f:
            f.write(transcribed_text)
        print("\n✅ Result saved to transcription_result.txt")

    except Exception as e:
        print(f"❌ Error during transcription: {e}")

    # Cleanup
    if os.path.exists(audio_path) and audio_path != "test_audio.wav":
        os.remove(audio_path)
        print("🧹 Temporary file cleaned up")
else:
    print("⚠️ No audio file available for transcription")

print("\n" + "=" * 50)
print("Powered by Vosk & OpenAI Whisper • Running 100% locally")


🎙️ Live Audio Transcriber
Record your audio and transcribe it using Vosk or Whisper.

🔄 Select Transcription Model:
1. Vosk (Fast, Offline)
2. Whisper (High Quality)
Enter choice (1-2, default: 2): 2

⏳ Transcribing...


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


✅ Using Whisper Model

📝 Transcribed Text:
--------------------------------------------------
 Here are a few ways to say these phrases in English. Greetings and well wishes. Hi, hello, good morning. Hope you have a wonderful, productive day. Wishing you a great day.
--------------------------------------------------

✅ Result saved to transcription_result.txt
🧹 Temporary file cleaned up

Powered by Vosk & OpenAI Whisper • Running 100% locally
